In [13]:
# ==========================================
# CELDA 1: Extracción de Datos (ETL Base)
# ==========================================
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# 1. Conexión a la Base de Datos
user = "Joasro"
password = "Akriila123." 
host = "localhost"
db = "dss_academico_unah"

engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{db}")

print("Extrayendo datos de la universidad...")

# Extraemos las tablas clave
df_historial = pd.read_sql("""
    SELECT h.Hash_Cuenta, h.ID_Clase, h.Estado, h.Periodo_Cursado, 
           m.Unidades_Valorativas, m.Codigo_Oficial, m.Plan_Perteneciente
    FROM Historial_Academico h
    JOIN Malla_Curricular m ON h.ID_Clase = m.ID_Clase
""", engine)

df_estudiantes = pd.read_sql("SELECT Hash_Cuenta, Plan_Estudio, Ano_Ingreso FROM Estudiantes", engine)
df_malla = pd.read_sql("SELECT ID_Clase, Codigo_Oficial, Nombre_Clase, Unidades_Valorativas, Plan_Perteneciente, Prerrequisitos, ID_Area FROM Malla_Curricular", engine)

# 🛑 EL FIX CRÍTICO ESTÁ AQUÍ: Añadimos Jornada_Preferencia a la extracción
df_censo = pd.read_sql("SELECT Hash_Cuenta, ID_Clase, Jornada_Preferencia FROM censo_periodo_actual", engine)

print(f"Datos extraídos: {len(df_historial)} registros históricos, {len(df_estudiantes)} estudiantes activos.")

Extrayendo datos de la universidad...
Datos extraídos: 4769 registros históricos, 135 estudiantes activos.


In [14]:
# ==========================================
# CELDA 2: Funciones de Dominio Académico
# ==========================================

EQUIVALENCIAS_REVERSAS = {
    'IS-110': 'ISC-101', 'MM-314': 'ISC-101', 'IS-210': 'ISC-102', 
    'IS-310': 'ISC-211', 'IS-410': 'ISC-103', 'IS-501': 'ISC-321',
    'IS-601': 'ISC-422', 'IS-602': 'ISC-341', 'IS-702': 'ISC-306', 
    'IS-311': 'IE-326',  'IS-510': 'IE-326',  'IS-511': 'ISC-331',
    'IS-611': 'ISC-332', 'IS-412': 'ISC-333', 'IS-512': 'ISC-334', 
    'IS-115': 'ISC-552', 'IS-802': 'ISC-408'
}

def cumple_prerrequisitos(req_text, aprobadas_set, uv_est):
    """Evalúa si el alumno cumple los requisitos para llevar una clase."""
    if not req_text or str(req_text).lower() in ['ninguno', 'nan', 'null', '']:
        return True
    if "140" in str(req_text):
        return uv_est >= 140
    reqs = [r.strip().upper() for r in str(req_text).split(',')]
    return all(r in aprobadas_set for r in reqs)

def extender_aprobadas(aprobadas_base):
    """Si aprobó una clase vieja, se asume que aprobó su equivalente nueva."""
    extendidas = set(aprobadas_base)
    for vieja, nueva in EQUIVALENCIAS_REVERSAS.items():
        if vieja in extendidas:
            extendidas.add(nueva)
    return extendidas

# 🛑 NUEVA LÓGICA: SÍNTESIS DE OPTATIVAS PARA EGRESANDOS
OPTATIVAS_2021 = ['IS-910', 'IS-911', 'IS-914', 'IS-912', 'IS-913']

def calcular_es_egresando(aprobadas_set, plan_estudiante, df_malla):
    """Calcula si al estudiante le faltan 8 o menos clases, saturando las optativas."""
    malla_carrera = df_malla[
        (df_malla['Plan_Perteneciente'] == plan_estudiante) & 
        (df_malla['Codigo_Oficial'].str.startswith(('IS', 'ISC', 'IE')))
    ]
    
    if plan_estudiante == '2021':
        malla_core = malla_carrera[~malla_carrera['Codigo_Oficial'].isin(OPTATIVAS_2021)]
        aprobadas_core = len([c for c in aprobadas_set if c in malla_core['Codigo_Oficial'].values])
        core_faltantes = len(malla_core) - aprobadas_core
        
        aprobadas_optativas = len([c for c in aprobadas_set if c in OPTATIVAS_2021])
        optativas_faltantes = max(0, 3 - aprobadas_optativas)
        
        total_faltantes = core_faltantes + optativas_faltantes
    else:
        # Para el Plan 2025, cuenta clases faltantes de forma lineal
        aprobadas_carrera = len([c for c in aprobadas_set if c in malla_carrera['Codigo_Oficial'].values])
        total_faltantes = len(malla_carrera) - aprobadas_carrera
        
    return 1 if total_faltantes <= 8 else 0

In [15]:
# ==========================================
# CELDA 3: Construcción del Set de Entrenamiento Histórico (Filtrado por Carrera)
# ==========================================
print("Reconstruyendo el pasado para entrenar a la IA (Solo clases de Sistemas)...")

training_data = []

# Ordenamos los periodos cronológicamente
periodos = sorted(df_historial['Periodo_Cursado'].unique(), 
                  key=lambda x: (int(x.split('-')[1]), int(x.split('-')[0])))

dict_estudiantes = df_estudiantes.set_index('Hash_Cuenta').to_dict('index')

for i in range(1, len(periodos)):
    periodo_actual = periodos[i]
    historia_previa = df_historial[df_historial['Periodo_Cursado'].apply(lambda x: periodos.index(x) < i)]
    matriculas_actuales = df_historial[df_historial['Periodo_Cursado'] == periodo_actual]

    for hash_est, info in dict_estudiantes.items():
        plan = info['Plan_Estudio']
        hist_estudiante = historia_previa[historia_previa['Hash_Cuenta'] == hash_est]
        if hist_estudiante.empty: continue 
            
        aprobadas_antes = set(hist_estudiante[hist_estudiante['Estado'] == 'Aprobado']['Codigo_Oficial'])
        aprobadas_antes = extender_aprobadas(aprobadas_antes)
        uv_antes = hist_estudiante[hist_estudiante['Estado'] == 'Aprobado']['Unidades_Valorativas'].sum()
        
        clases_matriculadas = set(matriculas_actuales[matriculas_actuales['Hash_Cuenta'] == hash_est]['Codigo_Oficial'])
        
        # 🔥 EL FILTRO MÁGICO: Solo tomamos clases del plan que empiecen con IS, ISC o IE
        clases_plan = df_malla[
            (df_malla['Plan_Perteneciente'] == plan) & 
            (df_malla['Codigo_Oficial'].str.startswith(('IS', 'ISC', 'IE')))
        ]
        
        for _, clase in clases_plan.iterrows():
            cod = clase['Codigo_Oficial']
            if cod in aprobadas_antes: continue 
                
            if cumple_prerrequisitos(clase['Prerrequisitos'], aprobadas_antes, uv_antes):
                training_data.append({
                    'UV_Acumuladas': uv_antes,
                    # 🛑 AQUÍ SE APLICA LA MAGIA DE LA NUEVA LÓGICA DE OPTATIVAS:
                    'Es_Egresando': calcular_es_egresando(aprobadas_antes, plan, df_malla),
                    'Plan_n': 1 if plan == '2025' else 0,
                    'Matriculo_Real': 1 if cod in clases_matriculadas else 0 
                })

df_train = pd.DataFrame(training_data)
print(f"✅ Set de entrenamiento creado: {len(df_train)} escenarios históricos analizados.")
display(df_train['Matriculo_Real'].value_counts())

Reconstruyendo el pasado para entrenar a la IA (Solo clases de Sistemas)...
✅ Set de entrenamiento creado: 3809 escenarios históricos analizados.


Matriculo_Real
1    2137
0    1672
Name: count, dtype: int64

In [16]:
# ==========================================
# CELDA 4: Entrenamiento del Motor Predictivo
# ==========================================
from sklearn.model_selection import train_test_split

X = df_train[['UV_Acumuladas', 'Es_Egresando', 'Plan_n']]
y = df_train['Matriculo_Real']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_base = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42,
    eval_metric='logloss'
)

print("Entrenando red neuronal XGBoost...")
modelo_base.fit(X_train, y_train)

y_pred = modelo_base.predict(X_test)
print(f"Precisión del modelo base (Solo con historial): {accuracy_score(y_test, y_pred):.2f}")

Entrenando red neuronal XGBoost...
Precisión del modelo base (Solo con historial): 0.65


In [17]:
# ==========================================
# CELDA 5: Predicción IPAC-2026 (Demanda 1:1 Real) + Hora del Censo
# ==========================================
import numpy as np

print("Calculando demanda real basada estrictamente en la base de datos...")

prediccion_actual = []
# Pre-calcular aprobadas para no recalcular en el bucle
aprobadas_totales = df_historial[df_historial['Estado'] == 'Aprobado'].groupby('Hash_Cuenta')['Codigo_Oficial'].apply(set).to_dict()

for hash_est, info in dict_estudiantes.items():
    plan = info['Plan_Estudio']
    aprobadas = extender_aprobadas(aprobadas_totales.get(hash_est, set()))
    
    # Calcular UV actuales
    uv_actuales = sum([df_malla[df_malla['Codigo_Oficial'] == c]['Unidades_Valorativas'].values[0] for c in aprobadas if c in df_malla['Codigo_Oficial'].values])
    
    clases_plan = df_malla[(df_malla['Plan_Perteneciente'] == plan) & (df_malla['Codigo_Oficial'].str.startswith(('IS', 'ISC', 'IE')))]
    
    for _, clase in clases_plan.iterrows():
        cod = clase['Codigo_Oficial']
        id_clase = clase['ID_Clase']
        
        if cod in aprobadas:
            continue
            
        if cumple_prerrequisitos(clase['Prerrequisitos'], aprobadas, uv_actuales):
            en_censo = 1 if ((df_censo['Hash_Cuenta'] == hash_est) & (df_censo['ID_Clase'] == id_clase)).any() else 0
            
            prediccion_actual.append({
                'Hash_Cuenta': hash_est,
                'ID_Clase': id_clase,
                'Codigo_Oficial': cod,
                'Nombre_Clase': clase['Nombre_Clase'],
                'ID_Area': clase['ID_Area'],         
                'UV_Acumuladas': uv_actuales,
                'Es_Egresando': calcular_es_egresando(aprobadas, plan, df_malla),
                'Plan_n': 1 if plan == '2025' else 0,
                'En_Censo': en_censo
            })

df_ipac2026 = pd.DataFrame(prediccion_actual)

# 1. Predicción Base (Manteniendo tu lógica original XGBoost)
X_ipac = df_ipac2026[['UV_Acumuladas', 'Es_Egresando', 'Plan_n']]
df_ipac2026['Probabilidad_ML'] = modelo_base.predict_proba(X_ipac)[:, 1]

# 2. Multiplicador por Censo Actual
df_ipac2026['Probabilidad_Final'] = np.clip(df_ipac2026['Probabilidad_ML'] + (df_ipac2026['En_Censo'] * 0.35), 0, 1)

# 3. SUMA EXACTA DE LA DEMANDA REAL (SIN FACTOR DE EXPANSIÓN)
demanda_final = df_ipac2026.groupby(['ID_Clase', 'Codigo_Oficial', 'Nombre_Clase', 'ID_Area'])['Probabilidad_Final'].sum().reset_index()

# Redondeamos hacia arriba para asegurar el cupo (Ej: si la prob suma 1.2, reservamos 2 cupos)
demanda_final['Cupos_Estimados'] = np.ceil(demanda_final['Probabilidad_Final']).astype(int)

# Filtro de clases sin demanda real
demanda_final = demanda_final[demanda_final['Cupos_Estimados'] > 0]

# ==========================================
# 4. EXTRACCIÓN DE LA HORA EXACTA MÁS VOTADA
# ==========================================
if not df_censo.empty and 'Jornada_Preferencia' in df_censo.columns:
    horas_censo = df_censo.groupby(['ID_Clase', 'Jornada_Preferencia'], as_index=False).size()
    horas_censo.rename(columns={'size': 'Votos'}, inplace=True)
    
    top_horas = horas_censo.sort_values('Votos', ascending=False).drop_duplicates(subset=['ID_Clase'])
    top_horas.rename(columns={'Jornada_Preferencia': 'Hora_Sugerida'}, inplace=True)

    demanda_final = demanda_final.merge(top_horas[['ID_Clase', 'Hora_Sugerida']], on='ID_Clase', how='left')
    demanda_final['Hora_Sugerida'] = demanda_final['Hora_Sugerida'].fillna('Sin preferencia')
else:
    demanda_final['Hora_Sugerida'] = 'Sin preferencia'

demanda_final = demanda_final.sort_values(by='Cupos_Estimados', ascending=False)
demanda_final = demanda_final[['ID_Clase', 'Codigo_Oficial', 'Nombre_Clase', 'ID_Area', 'Cupos_Estimados', 'Hora_Sugerida']]

# ==========================================
# NUEVO: LÓGICA DE NUEVO INGRESO (ISC-101)
# ==========================================
try:
    # 1. Análisis de Tendencia de la PAA (Desde tu CSV histórico)
    df_hist_ing = pd.read_csv('../data/historial_ingresos.csv')
    col_ing = 'Primer Ingreso' if 'Primer Ingreso' in df_hist_ing.columns else 'Primer_Ingreso'
    df_validos = df_hist_ing[df_hist_ing[col_ing] > 0].dropna(subset=[col_ing])
    
    # Suavizado exponencial: 50% de peso al periodo más reciente (39 alumnos)
    ultimos = df_validos[col_ing].tail(3).values
    if len(ultimos) == 3:
        novatos_proyectados = int(np.ceil((ultimos[2]*0.5) + (ultimos[1]*0.3) + (ultimos[0]*0.2)))
    else:
        novatos_proyectados = int(np.ceil(np.mean(ultimos))) if len(ultimos) > 0 else 35

    # 2. Tasa de Absorción Real (Generación 2023+ de tu base de datos)
    df_mod = df_estudiantes[df_estudiantes['Ano_Ingreso'] >= 2023]
    hashes_str = tuple(df_mod['Hash_Cuenta'].tolist())
    q_abs = f"SELECT COUNT(DISTINCT h.Hash_Cuenta) as t FROM Historial_Academico h JOIN Malla_Curricular m ON h.ID_Clase = m.ID_Clase WHERE m.Codigo_Oficial='ISC-101' AND h.Hash_Cuenta IN {hashes_str}"
    # Obtenemos qué porcentaje de los ingresados realmente llevaron la clase en su 1er periodo
    tasa_real = min(max(pd.read_sql(q_abs, engine)['t'].iloc[0] / len(df_mod), 0.50), 1.0)
    
    cupos_novatos = int(np.ceil(novatos_proyectados * tasa_real))
    print(f"📈 [Exógeno] Proyectados {cupos_novatos} cupos obligatorios para ISC-101.")
    
    # 3. Inyectar o sumar a la demanda final
    if 'ISC-101' in demanda_final['Codigo_Oficial'].values:
        idx = demanda_final.index[demanda_final['Codigo_Oficial'] == 'ISC-101'][0]
        demanda_final.at[idx, 'Cupos_Estimados'] += cupos_novatos
    else:
        # Si ningún alumno de reingreso la necesitaba, la creamos para los novatos
        info_i = df_malla[df_malla['Codigo_Oficial'] == 'ISC-101'].iloc[0]
        demanda_final = pd.concat([demanda_final, pd.DataFrame([{
            'ID_Clase': info_i['ID_Clase'], 'Codigo_Oficial': 'ISC-101', 'Nombre_Clase': info_i['Nombre_Clase'], 
            'ID_Area': info_i['ID_Area'], 'Cupos_Estimados': cupos_novatos, 'Hora_Sugerida': 'Sin preferencia'
        }])], ignore_index=True)
except Exception as e:
    print(f"⚠️ Aviso: Error en predicción exógena ({e})")

# Mostrar y Guardar
pd.set_option('display.max_rows', None)
print(f"🎯 RESULTADO FINAL: Demanda Estricta IPAC-2026")
print(f"📊 Total de estudiantes procesados en BD: {len(dict_estudiantes)}")
print("-" * 60)
display(demanda_final)
pd.reset_option('display.max_rows')

demanda_final.to_csv('../data/demanda_proyectada_2026.csv', index=False)
print("✅ Archivo 'demanda_proyectada_2026.csv' guardado con la demanda 1:1 real.")

Calculando demanda real basada estrictamente en la base de datos...
📈 [Exógeno] Proyectados 16 cupos obligatorios para ISC-101.
🎯 RESULTADO FINAL: Demanda Estricta IPAC-2026
📊 Total de estudiantes procesados en BD: 135
------------------------------------------------------------


,ID_Clase,Codigo_Oficial,Nombre_Clase,ID_Area,Cupos_Estimados,Hora_Sugerida
27,54,IS-911,Microprocesadores,10.0,10,Sin preferencia
19,46,IS-701,Inteligencia Artificial,7.0,7,Sin preferencia
16,43,IS-702,Análisis y Diseño de Sistemas,4.0,6,Sin preferencia
26,53,IS-910,Teoría de la Simulación,7.0,6,Sin preferencia
29,56,IS-913,Diseño de Compiladores,1.0,6,Sin preferencia
17,44,IS-721,Contabilidad I,5.0,5,Sin preferencia
11,37,IS-611,Redes de Datos II,21.0,5,Sin preferencia
12,38,IS-711,Diseño Digital,10.0,4,Sin preferencia
35,134,ISC-204,Paradigmas de Programación,12.0,4,Sin preferencia
22,49,IS-902,Industria del Software,4.0,4,Sin preferencia


✅ Archivo 'demanda_proyectada_2026.csv' guardado con la demanda 1:1 real.


In [18]:
# Guardar resultados para el optimizador
demanda_final.to_csv('../data/demanda_proyectada_2026.csv', index=False)
print("✅ Archivo 'demanda_proyectada_2026.csv' guardado en la carpeta /data.")

✅ Archivo 'demanda_proyectada_2026.csv' guardado en la carpeta /data.
